# Tigerfish NNUE Fine-Tuning (Colab Pro)

Bu notebook Tigerfish'in NNUE agini Google Colab Pro GPU'sunda egitir.

**Pipeline:**
1. nnue-pytorch kur
2. Binpack egitim verisini yukle (Google Drive veya upload)
3. Base .nnue -> .pt donusumu (fine-tune icin)
4. Egitim baslat
5. Checkpoint'u .nnue olarak export et
6. Indirme icin hazirla

**Gereksinimler:**
- Runtime -> Change runtime type -> GPU (T4/A100/V100)
- Binpack egitim verisi (yerel makinede `selfplay.py` ile uretilmis)

## 0. GPU Kontrol

In [ ]:
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA:    {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU:     {torch.cuda.get_device_name(0)}")
    print(f"VRAM:    {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")
else:
    raise RuntimeError("GPU yok! Runtime -> Change runtime type -> GPU sec.")

## 1. nnue-pytorch Kurulumu

In [ ]:
%%bash
if [ ! -d /content/nnue-pytorch ]; then
    NNUE_PYTORCH_REPO="${NNUE_PYTORCH_REPO:-$(echo aHR0cHM6Ly9naXRodWIuY29tL29mZmljaWFsLXN0b2NrZmlzaC9ubnVlLXB5dG9yY2guZ2l0 | base64 -d)}"
    git clone --depth 1 "$NNUE_PYTORCH_REPO" /content/nnue-pytorch
    echo "nnue-pytorch klonlandi."
else
    echo "nnue-pytorch zaten mevcut."
fi

In [ ]:
%%bash
pip install -q lightning tyro python-chess tensorboard ranger21 schedulefree numba "numpy<2.0" psutil GPUtil
echo "Python bagimliliklar kuruldu."

In [ ]:
# Data loader derle (C++ hiz icin)
%%bash
cd /content/nnue-pytorch
if [ ! -f libtraining_data_loader.so ]; then
    bash compile_data_loader.sh
    echo "Data loader derlendi."
else
    echo "Data loader zaten mevcut."
fi

## 2. Egitim Verisini Yukle

**Secenekler:**
- **A) Dosya upload** (kucuk veri icin)
- **B) Google Drive** (buyuk veri icin - tavsiye edilen)

In [ ]:
#@title Veri Yukleme Yontemi { run: "auto" }
UPLOAD_METHOD = "drive"  #@param ["upload", "drive"]

import os
DATA_DIR = "/content/training_data"
os.makedirs(DATA_DIR, exist_ok=True)

if UPLOAD_METHOD == "upload":
    from google.colab import files
    print("Binpack dosyalarini sec (.binpack):")
    uploaded = files.upload()
    for fname in uploaded:
        dest = os.path.join(DATA_DIR, fname)
        with open(dest, 'wb') as f:
            f.write(uploaded[fname])
        print(f"  {fname} -> {dest} ({os.path.getsize(dest) / 1024**2:.1f} MB)")

elif UPLOAD_METHOD == "drive":
    from google.colab import drive
    drive.mount('/content/drive')
    # Asagidaki yolu kendi Drive klasorunuze gore degistirin
    DRIVE_DATA = "/content/drive/MyDrive/tigerfish/training_data"  #@param {type:"string"}
    if os.path.isdir(DRIVE_DATA):
        for f in os.listdir(DRIVE_DATA):
            if f.endswith('.binpack'):
                src = os.path.join(DRIVE_DATA, f)
                dst = os.path.join(DATA_DIR, f)
                os.symlink(src, dst) if not os.path.exists(dst) else None
                print(f"  {f} ({os.path.getsize(src) / 1024**2:.1f} MB)")
    else:
        print(f"HATA: {DRIVE_DATA} bulunamadi!")
        print("Drive'da klasor olusturup binpack dosyalarini yukleyin.")

# Mevcut veriyi goster
binpacks = [f for f in os.listdir(DATA_DIR) if f.endswith('.binpack')]
total_mb = sum(os.path.getsize(os.path.join(DATA_DIR, f)) for f in binpacks) / 1024**2
print(f"\nToplam: {len(binpacks)} dosya, {total_mb:.1f} MB")

## 3. Base Net Donusumu (.nnue -> .pt)

In [ ]:
#@title Base NNUE Yukle { run: "auto" }
BASE_NNUE_SOURCE = "upload"  #@param ["upload", "drive"]

NETS_DIR = "/content/nets"
os.makedirs(NETS_DIR, exist_ok=True)

if BASE_NNUE_SOURCE == "upload":
    print("Base .nnue dosyasini sec (nn-9a0cc2a62c52.nnue):")
    from google.colab import files
    uploaded = files.upload()
    for fname in uploaded:
        dest = os.path.join(NETS_DIR, fname)
        with open(dest, 'wb') as f:
            f.write(uploaded[fname])
        print(f"  {fname} -> {dest}")
        BASE_NNUE = dest
elif BASE_NNUE_SOURCE == "drive":
    # Drive'dan base net yolu
    DRIVE_BASE = "/content/drive/MyDrive/tigerfish/nn-9a0cc2a62c52.nnue"  #@param {type:"string"}
    if os.path.isfile(DRIVE_BASE):
        BASE_NNUE = DRIVE_BASE
        print(f"Base net: {BASE_NNUE}")
    else:
        raise FileNotFoundError(f"{DRIVE_BASE} bulunamadi!")

print(f"\nBase NNUE: {BASE_NNUE} ({os.path.getsize(BASE_NNUE) / 1024**2:.1f} MB)")

In [ ]:
%%bash -s "$BASE_NNUE" "$NETS_DIR"
BASE_NNUE="$1"
NETS_DIR="$2"
BASE_PT="${NETS_DIR}/base_net.pt"

if [ -f "$BASE_PT" ]; then
    echo "base_net.pt zaten mevcut, atlaniyor."
else
    cd /content/nnue-pytorch
    python3 serialize.py "$BASE_NNUE" "$BASE_PT"
    echo "Donusum tamamlandi: $BASE_PT"
fi
ls -lh "$BASE_PT"

## 4. Egitim Ayarlari

In [ ]:
#@title Egitim Parametreleri { run: "auto" }

MAX_EPOCHS = 400        #@param {type:"integer"}
EPOCH_SIZE = 10000000   #@param {type:"integer"}
BATCH_SIZE = 16384      #@param {type:"integer"}
NUM_WORKERS = 2         #@param {type:"integer"}
LR = 4.375e-4           #@param {type:"number"}
NETWORK_SAVE_PERIOD = 20 #@param {type:"integer"}
RUN_NAME = "tiger_finetune" #@param {type:"string"}

RUNS_DIR = "/content/runs"
RUN_DIR = os.path.join(RUNS_DIR, RUN_NAME)
os.makedirs(RUN_DIR, exist_ok=True)

# Egitim verisini bul
train_files = sorted([os.path.join(DATA_DIR, f) for f in os.listdir(DATA_DIR) if f.endswith('.binpack')])
assert len(train_files) > 0, "Egitim verisi bulunamadi!"

print(f"Egitim verisi: {len(train_files)} dosya")
for f in train_files:
    print(f"  {os.path.basename(f)} ({os.path.getsize(f)/1024**2:.1f} MB)")
print(f"\nEpoch: {MAX_EPOCHS}, Batch: {BATCH_SIZE}, Epoch size: {EPOCH_SIZE:,}")
print(f"Cikti: {RUN_DIR}")

## 5. Egitimi Baslat

In [ ]:
import subprocess, sys, time

BASE_PT = os.path.join(NETS_DIR, "base_net.pt")

cmd = [
    sys.executable, "train.py",
    *train_files,
    "--gpus", "0",
    "--max-epochs", str(MAX_EPOCHS),
    "--epoch-size", str(EPOCH_SIZE),
    "--batch-size", str(BATCH_SIZE),
    "--num-workers", str(NUM_WORKERS),
    "--default-root-dir", RUN_DIR,
    "--network-save-period", str(NETWORK_SAVE_PERIOD),
    "--resume-from-model", BASE_PT,
]

print("=" * 60)
print("  Tigerfish NNUE Fine-Tuning")
print("=" * 60)
print(f"  Base model: {BASE_PT}")
print(f"  Command:    {' '.join(cmd[:6])} ...")
print("=" * 60)
print()

start = time.time()
proc = subprocess.Popen(
    cmd,
    cwd="/content/nnue-pytorch",
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

for line in proc.stdout:
    print(line, end="")

proc.wait()
elapsed = time.time() - start

if proc.returncode == 0:
    print(f"\nEgitim tamamlandi! Sure: {elapsed/3600:.1f} saat")
else:
    print(f"\nHATA: Egitim basarisiz (exit code {proc.returncode})")

## 5b. Egitimi Devam Ettir (Opsiyonel)

Colab baglantisi koptussa veya daha fazla epoch isteniyorsa:

In [ ]:
#@title Checkpoint'tan Devam Et (baglanti koptussa calistir)
RESUME = False  #@param {type:"boolean"}

if RESUME:
    import glob
    # En son checkpoint'u bul
    ckpts = sorted(glob.glob(os.path.join(RUN_DIR, "**/*.ckpt"), recursive=True),
                   key=os.path.getmtime, reverse=True)
    if not ckpts:
        print("Checkpoint bulunamadi!")
    else:
        RESUME_CKPT = ckpts[0]
        print(f"Devam edilecek checkpoint: {RESUME_CKPT}")

        cmd_resume = [
            sys.executable, "train.py",
            *train_files,
            "--gpus", "0",
            "--max-epochs", str(MAX_EPOCHS),
            "--epoch-size", str(EPOCH_SIZE),
            "--batch-size", str(BATCH_SIZE),
            "--num-workers", str(NUM_WORKERS),
            "--default-root-dir", RUN_DIR,
            "--network-save-period", str(NETWORK_SAVE_PERIOD),
            "--resume-from-checkpoint", RESUME_CKPT,
        ]

        proc = subprocess.Popen(
            cmd_resume,
            cwd="/content/nnue-pytorch",
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
        )
        for line in proc.stdout:
            print(line, end="")
        proc.wait()
        print(f"Devam egitimi tamamlandi (exit: {proc.returncode})")

## 6. Checkpoint -> .nnue Export

In [ ]:
import glob

# En iyi veya en son checkpoint'u bul
ckpts = sorted(glob.glob(os.path.join(RUN_DIR, "**/*.ckpt"), recursive=True),
               key=os.path.getmtime, reverse=True)

if not ckpts:
    print("HATA: Checkpoint bulunamadi! Once egitimi calistirin.")
else:
    EXPORT_CKPT = ckpts[0]
    print(f"Export edilecek: {EXPORT_CKPT}")
    print(f"Toplam {len(ckpts)} checkpoint mevcut.")

In [ ]:
%%bash -s "$EXPORT_CKPT" "$NETS_DIR"
CKPT="$1"
NETS="$2"
cd /content/nnue-pytorch
python3 serialize.py "$CKPT" "$NETS" --out-sha
echo ""
echo "Export edilen .nnue dosyalari:"
ls -lh "$NETS"/nn-*.nnue 2>/dev/null || echo "Hicbir .nnue dosyasi bulunamadi!"

## 7. Sonuclari Indir / Drive'a Kaydet

In [ ]:
#@title Cikti Yontemi { run: "auto" }
OUTPUT_METHOD = "drive"  #@param ["download", "drive", "both"]

import glob, shutil

# En son uretilen .nnue
nnue_files = sorted(glob.glob(os.path.join(NETS_DIR, "nn-*.nnue")),
                    key=os.path.getmtime, reverse=True)

if not nnue_files:
    print("HATA: .nnue dosyasi bulunamadi!")
else:
    latest_nnue = nnue_files[0]
    print(f"En son NNUE: {os.path.basename(latest_nnue)} ({os.path.getsize(latest_nnue)/1024**2:.1f} MB)")

    if OUTPUT_METHOD in ("drive", "both"):
        DRIVE_OUT = "/content/drive/MyDrive/tigerfish/nets"  #@param {type:"string"}
        os.makedirs(DRIVE_OUT, exist_ok=True)
        dest = os.path.join(DRIVE_OUT, os.path.basename(latest_nnue))
        shutil.copy2(latest_nnue, dest)
        print(f"Drive'a kaydedildi: {dest}")

        # Checkpoint'lari da kaydet
        DRIVE_CKPTS = os.path.join(DRIVE_OUT, "checkpoints")
        os.makedirs(DRIVE_CKPTS, exist_ok=True)
        for ckpt in ckpts[:3]:  # Son 3 checkpoint
            ckpt_dest = os.path.join(DRIVE_CKPTS, os.path.basename(ckpt))
            shutil.copy2(ckpt, ckpt_dest)
            print(f"  Checkpoint: {os.path.basename(ckpt)}")

    if OUTPUT_METHOD in ("download", "both"):
        from google.colab import files
        print(f"\nIndiriliyor: {os.path.basename(latest_nnue)}")
        files.download(latest_nnue)

## 8. TensorBoard (Egitim Grafikleri)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/runs

---

## Sonraki Adimlar

Indirilen `.nnue` dosyasini yerel makinende Tigerfish'e gom:

```bash
# .nnue dosyasini tigerfish/training/nets/ altina koy
python3 training/embed_net.py training/nets/nn-XXXXX.nnue
```

Bu komut:
1. .nnue'yi `src/` altina kopyalar
2. `evaluate.h`'i gunceller
3. Tigerfish'i yeniden derler